In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [3]:
user_id = "user_001"
application_context = "personal_assistant"

# namespace 정의
namespace = (user_id, application_context)

In [4]:
store.put(
    namespace,
    "memory_001",
    {
        "facts": [
            "사용자는 아아를 선호함",
            "사용자는 매일 아침 7시에 일어남",
        ],
        "language": "Korean",
    },
)

In [5]:
item = store.get(namespace, "memory_001")
print("저장된 메모리:", item.value)

저장된 메모리: {'facts': ['사용자는 아아를 선호함', '사용자는 매일 아침 7시에 일어남'], 'language': 'Korean'}


In [6]:
items = store.search(namespace)

In [7]:
items

[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아를 선호함', '사용자는 매일 아침 7시에 일어남'], 'language': 'Korean'}, created_at='2026-05-24T04:51:20.601175+00:00', updated_at='2026-05-24T04:51:20.601175+00:00', score=None)]

In [8]:
store.put(
    namespace,
    "memory_002",
    {
        "facts": [
            "사용자는 랭체인 공부를 좋아함",
            "학습용 챗봇 프로젝트를 진행 중",
        ]
    },
)

In [9]:
item = store.get(namespace, "memory_002")
print("저장된 메모리:", item.value)

저장된 메모리: {'facts': ['사용자는 랭체인 공부를 좋아함', '학습용 챗봇 프로젝트를 진행 중']}


In [10]:
items = store.search(namespace)

In [11]:
items

[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아를 선호함', '사용자는 매일 아침 7시에 일어남'], 'language': 'Korean'}, created_at='2026-05-24T04:51:20.601175+00:00', updated_at='2026-05-24T04:51:20.601175+00:00', score=None),
 Item(namespace=['user_001', 'personal_assistant'], key='memory_002', value={'facts': ['사용자는 랭체인 공부를 좋아함', '학습용 챗봇 프로젝트를 진행 중']}, created_at='2026-05-24T04:51:20.637668+00:00', updated_at='2026-05-24T04:51:20.637668+00:00', score=None)]

In [12]:
from dataclasses import dataclass

# 실행 컨텍스트 정의 (누가 실행하는지 식별)
@dataclass
class Context:
    user_id: str
    app_name: str

In [13]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_memory(request, handler):
    current_user = request.runtime.context.user_id
    current_app = request.runtime.context.app_name
    memories = request.runtime.store.search((current_user, current_app))

    memory_content = "기록된 정보 없음"
    if memories:
        # 검색된 메모리들을 텍스트 변환
        extracted_facts = []
        for item in memories:
            if "facts" in item.value:
                extracted_facts.extend(item.value["facts"])
        memory_content = "\n- ".join(extracted_facts)

    system_message=f"사용자 관련 장기 메모리 : {memory_content}"
    request = request.override(system_prompt=system_message)
    return handler(request)

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    store=store, # store 연결
    context_schema=Context,
    middleware=[inject_memory] # 미들웨어 장착
)

In [19]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

In [ ]:
response

{'messages': [HumanMessage(content='나에 대해 알고 있는 모든 것을 말해줘', additional_kwargs={}, response_metadata={}, id='70f86d98-8660-46b1-9e12-40171771bb7f'),
  AIMessage(content='네, 사용자님에 대해 제가 기억하고 있는 내용은 다음과 같습니다.\n\n*   **아이스 아메리카노(아아)를 선호하십니다.**\n*   **매일 아침 7시에 일어나십니다.**\n*   **랭체인(LangChain) 공부를 좋아하십니다.**\n*   **현재 학습용 챗봇 프로젝트를 진행 중이십니다.**\n\n이 정보들을 바탕으로 앞으로 사용자님께 더 나은 지원을 해드릴 수 있도록 노력하겠습니다!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c141f-60df-7700-a18a-71ef1b7d7564-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 899, 'total_tokens': 963, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 791}})]}